[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YamKhoonTor/datums-book/blob/main/notebooks/Chapter_04_Heights_and_vertical_datums.ipynb)

Companion notebook to *Datums, Coordinate Systems and Map Projections* by Yam Khoon Tor. Choose *Runtime → Run all*, compare with the output stored beneath each listing, then change the values in the *Try it* cell and run it again.

# Chapter 4. Heights and vertical datums

Companion notebook to *Datums, Coordinate Systems and Map Projections: A Practical Guide for Surveyors and Engineers, with SVY21 Worked Examples*, by Yam Khoon Tor.

*Copyright © 2026 Yam Khoon Tor. All rights reserved. Readers who have bought the book may run, copy and modify this notebook for personal study and their own professional work; it may not be republished, distributed or sold without the author's written permission. Provided as a teaching aid, without warranty: verify results independently before use in real work.*

**How to use this notebook.** Run the cells in order from the top: select a cell and press the run button (or Shift+Enter), or choose *Run all* from the menu. Each listing is exactly as printed in the book. Beneath it the notebook stores the output of the run, and repeats the output printed in the book so that the two can be compared. If an unaltered notebook gives different numbers, something is wrong with the installation and should be put right before going on. The last section, **Try it**, repeats an example with the input values gathered at the top of the cell, to be changed freely.

Requirements: Python 3.9 or later and NumPy.

## Listing 4.1  Geometric geoid surface by least squares

In [1]:
import numpy as np

# Benchmarks: name, northing (m), easting (m), ellipsoidal height h (m),
# levelled reduced level H (m). Illustrative values.
BENCHMARKS = [
    ("BM1", 30500.0, 22400.0, 16.918,  7.132),
    ("BM2", 36200.0, 24800.0, 25.451, 15.640),
    ("BM3", 41800.0, 23100.0, 33.777, 23.862),
    ("BM4", 43300.0, 29700.0, 21.342, 11.562),
    ("BM5", 38100.0, 31200.0, 48.809, 39.115),
    ("BM6", 32600.0, 30100.0, 15.160,  5.523),
    ("BM7", 35400.0, 36900.0, 12.371,  2.871),
    ("BM8", 40600.0, 37600.0, 30.033, 20.469),
]
N0, E0 = 38744.572, 28001.642        # reference point of the surface


def fit_plane(benchmarks):
    """Least squares fit of N = c0 + c1*dn + c2*de, with dn, de in km."""
    A, L = [], []
    for _, n, e, h, H in benchmarks:
        A.append([1.0, (n - N0) / 1000.0, (e - E0) / 1000.0])
        L.append(h - H)                        # observed undulation
    A, L = np.array(A), np.array(L)
    x = np.linalg.solve(A.T @ A, A.T @ L)      # normal equations
    v = A @ x - L                              # residuals
    dof = len(L) - len(x)                      # degrees of freedom
    s0 = float(np.sqrt(v @ v / dof))           # standard deviation of fit
    return x, v, s0


def undulation(x, n, e):
    """Interpolated undulation at grid position (n, e)."""
    return x[0] + x[1] * (n - N0) / 1000.0 + x[2] * (e - E0) / 1000.0


if __name__ == "__main__":
    x, v, s0 = fit_plane(BENCHMARKS)
    print(f"c0 = {x[0]:.4f} m  c1 = {x[1]:.5f} m/km  c2 = {x[2]:.5f} m/km")
    for (name, n, e, h, H), vi in zip(BENCHMARKS, v):
        print(f"{name}  observed N = {h - H:.3f}  residual = {vi * 1000:+.1f} mm")
    print(f"standard deviation of fit = {s0 * 1000:.1f} mm")

    n, e, h = 37000.0, 28500.0, 27.818         # new point
    Nu = undulation(x, n, e)
    print(f"new point: N = {Nu:.3f} m  H = {h - Nu:.3f} m")

c0 = 9.7644 m  c1 = 0.01304 m/km  c2 = -0.02363 m/km
BM1  observed N = 9.786  residual = +3.2 mm
BM2  observed N = 9.811  residual = -4.2 mm
BM3  observed N = 9.915  residual = +5.0 mm
BM4  observed N = 9.780  residual = +3.6 mm
BM5  observed N = 9.694  residual = -13.6 mm
BM6  observed N = 9.637  residual = -2.3 mm
BM7  observed N = 9.500  residual = +10.5 mm
BM8  observed N = 9.564  residual = -2.2 mm
standard deviation of fit = 8.6 mm
new point: N = 9.730 m  H = 18.088 m


**Expected output, as printed in the book:**

```
c0 = 9.7644 m  c1 = 0.01304 m/km  c2 = -0.02363 m/km
BM1  observed N = 9.786  residual = +3.2 mm
BM2  observed N = 9.811  residual = -4.2 mm
BM3  observed N = 9.915  residual = +5.0 mm
BM4  observed N = 9.780  residual = +3.6 mm
BM5  observed N = 9.694  residual = -13.6 mm
BM6  observed N = 9.637  residual = -2.3 mm
BM7  observed N = 9.500  residual = +10.5 mm
BM8  observed N = 9.564  residual = -2.2 mm
standard deviation of fit = 8.6 mm
new point: N = 9.730 m  H = 18.088 m
```

## Try it

Change the values between the two dashed lines and run the cell again. Nothing here alters the listings above.

In [2]:
# ---- Change these values, then run the cell ----------------------
# A new point observed by GNSS: grid northing, easting, ellipsoidal height
n, e, h = 37000.0, 28500.0, 27.818
leave_out = None        # e.g. "BM5" to refit the surface without that benchmark
# -------------------------------------------------------------------

marks = [b for b in BENCHMARKS if b[0] != leave_out]
x, v, s0 = fit_plane(marks)
print(f"benchmarks used: {len(marks)}   standard deviation of fit = {s0 * 1000:.1f} mm")
Nu = undulation(x, n, e)
print(f"undulation N = {Nu:.3f} m")
print(f"height above the height datum H = h - N = {h - Nu:.3f} m")

benchmarks used: 8   standard deviation of fit = 8.6 mm
undulation N = 9.730 m
height above the height datum H = h - N = 18.088 m
